# Main text simulation study

This notebook implements the three main simulation subsections for the GRR manuscript. All table and plotting code is written directly in this notebook.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from inside the genriesz repository.")
    REPO_ROOT = parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import grr_ate, grr_att
from genriesz.basis import BaseBasis, TreatmentInteractionBasis
from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    CoverageDiagnosticBasis,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    generator_shift_for_estimand,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"
TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_LABELS = {
    "ra": "RA",
    "rw": "RW (IPW)",
    "arw": "ARW (AIPW)",
    "tmle": "TMLE",
    "SQ": "SQ-Riesz",
    "UKL": "UKL-Riesz",
    "BKL": "BKL-Riesz",
    "BP(0.5)": "BP-Riesz (omega = 0.5)",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random forest leaves",
    "rff": "Random Fourier features",
    "matching": "Nearest-neighbor matching",
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")
pd.options.display.max_rows = TABLE_CONFIG["max_rows"]


def label_of(value):
    """Return the display label for a stored result key."""

    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    """Replace stored result keys inside a composite display label."""

    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def prettify_labels(frame):
    """Return a copy with known result-key columns formatted for display."""

    out = frame.copy()
    for column in LABEL_COLUMNS:
        if column in out.columns:
            out[column] = out[column].map(label_of)
    return out


def display_table(frame, *, caption=None, digits=4):
    """Display a rounded table without changing the stored results."""

    table = prettify_labels(frame)
    numeric_columns = table.select_dtypes(include=[np.number]).columns
    table[numeric_columns] = table[numeric_columns].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


In [ ]:
# Publication-scale settings. Edit here if the manuscript grid changes.
N_REPLICATIONS = 100
SAMPLE_SIZE = 1200
DGP_NAMES = [
    "DGP 1: smooth heterogeneous effects",
    "DGP 2: weak overlap nonlinear design",
    "DGP 3: high-dimensional sparse confounding",
]
BASIS_KINDS_MAIN = ["rkhs", "polynomial", "rf"]
RIEZ_FEATURES = 120
POLYNOMIAL_DEGREE = 2
FOLDS = 5
RIEZ_LAMBDA_MAIN = 3e-1
OUTCOME_LAMBDA = 1e-3
RF_N_JOBS = 1  # Change to -1 for parallel RF fitting only if your environment does not emit repeated joblib warnings.

settings_table = pd.DataFrame([{
    "N_REPLICATIONS": N_REPLICATIONS,
    "SAMPLE_SIZE": SAMPLE_SIZE,
    "N_DGPS": len(DGP_NAMES),
    "BASIS_KINDS": ", ".join(BASIS_KINDS_MAIN),
    "RIEZ_FEATURES": RIEZ_FEATURES,
    "FOLDS": FOLDS,
    "RF_N_JOBS": RF_N_JOBS,
}])
display_table(settings_table, caption="Main simulation settings")


## 1. Compatible regressor-balancing loss-link pairs

Each loss uses its generator-induced compatible link. The balancing dictionary is treatment-interaction basis functions of the full regressor `X=(D,Z)`. This directly targets the score-relevant regression error for ATE under treatment-effect heterogeneity.

In [ ]:
# Direct Monte Carlo loop. This cell is intentionally written in the notebook so the experimental grid can be edited here.
rows = []
for dgp_name in DGP_NAMES:
    for rep in range(N_REPLICATIONS):
        data = make_simulation_data(dgp_name, n=SAMPLE_SIZE, seed=100000 + 1009 * rep)
        for basis_kind in BASIS_KINDS_MAIN:
            for loss_spec in COMPATIBLE_LOSSES:
                for estimand in ESTIMANDS:
                    fit_rows = fit_one_grr(
                        data,
                        estimand=estimand,
                        loss_spec=loss_spec,
                        basis_kind=basis_kind,
                        basis_mode="regressor",
                        cross_fit=True,
                        lam=RIEZ_LAMBDA_MAIN,
                        basis_features=RIEZ_FEATURES,
                        degree=POLYNOMIAL_DEGREE,
                        folds=FOLDS,
                        estimators=ESTIMATORS_ALL,
                        random_state=rep,
                    )
                    for row in fit_rows:
                        row["dgp"] = dgp_name
                        row["replication"] = rep
                    rows.extend(fit_rows)
main_compatible = pd.DataFrame(rows)
summary_compatible = summarize_estimates(main_compatible, ["dgp", "estimand", "basis", "loss", "estimator"])

# Display ATE and ATT in separate tables. Do not place them side by side.
for estimand_name in ESTIMANDS:
    table_df = summary_compatible[summary_compatible["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "basis", "loss", "estimator"])
    display_table(table_df, caption=f"Main simulation compatible loss-link pairs: {estimand_name}")


In [ ]:
# Box plots of ARW squared error. Edit titles, labels, colors, font sizes, line widths, and scale here.
plot_df = main_compatible[(main_compatible["status"] == "ok") & (main_compatible["estimator"] == "arw")].copy()
plot_df["method"] = plot_df["loss"] + " (" + plot_df["basis"] + ")"
plot_df["squared_error_plot"] = plot_df["squared_error"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for dgp_name in DGP_NAMES:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["dgp"] == dgp_name)].copy()
        if panel_df.empty:
            print(f"No plot data for {estimand_name} ({dgp_name}).")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        ordered_methods = sorted(panel_df["method"].unique())
        box_data = [panel_df.loc[panel_df["method"] == m, "squared_error_plot"].dropna().to_numpy() for m in ordered_methods]
        box = ax.boxplot(box_data, tick_labels=[prettify_method(_m) for _m in ordered_methods], widths=PLOT_CONFIG["box_width"], patch_artist=True, showfliers=False)
        for patch, method in zip(box["boxes"], ordered_methods):
            loss_name = method.split(" | ")[0]
            patch.set_facecolor(METHOD_COLORS.get(loss_name, "#CCCCCC"))
            patch.set_alpha(0.75)
        for median in box["medians"]:
            median.set_color("black")
            median.set_linewidth(1.2)
        ax.set_yscale(PLOT_CONFIG["squared_error_y_scale"])
        ax.set_title(f"Squared error in {estimand_name} estimation. {dgp_name}", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Loss and basis", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("Squared error", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="x", labelrotation=45, labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.tick_params(axis="y", labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
        fig.tight_layout()
        plt.show()


## 1b. External plug-in baseline: cross-fitted logistic IPW/AIPW

The baseline estimates the propensity score by a logistic regression on the raw covariates (numerically unpenalized) and plugs it into the IPW and AIPW forms. It uses the same stratified cross-fitting folds as the GRR arms, the same outcome-model specification as the RKHS GRR arms, and the same influence-function variance estimator, so the comparison isolates how the weights are estimated: plug-in propensity versus direct representer fitting. In the combined doubly robust table below this weights-only reading applies to the RKHS rows; the polynomial and random-forest GRR arms use their own outcome bases and are shown for context. Design record: paper-repo `doc/2026-07-17_baseline_experiments_design.md` (Phase 1).

In [ ]:
# External plug-in propensity baseline. Data are regenerated deterministically from
# the same seeds as the compatible loop above, and every fit is deterministic given
# random_state, so the GRR results above are unchanged.
plugin_rows = []
for dgp_name in DGP_NAMES:
    for rep in range(N_REPLICATIONS):
        data = make_simulation_data(dgp_name, n=SAMPLE_SIZE, seed=100000 + 1009 * rep)
        for estimand in ESTIMANDS:
            fit_rows = fit_one_plugin_logistic(
                data,
                estimand=estimand,
                folds=FOLDS,
                random_state=rep,
                basis_features=RIEZ_FEATURES,
            )
            for row in fit_rows:
                row["dgp"] = dgp_name
                row["replication"] = rep
            plugin_rows.extend(fit_rows)
main_plugin = pd.DataFrame(plugin_rows)
failed_plugin = main_plugin[main_plugin["status"] != "ok"]
assert failed_plugin.empty, failed_plugin  # never drop failures silently

summary_plugin = summarize_estimates(main_plugin, ["dgp", "estimand", "basis", "loss", "estimator"])
# summarize_estimates aggregates only its fixed diagnostic list; surface the
# propensity clip rate explicitly so a binding clip is visible.
plugin_clip = (
    main_plugin.groupby(["dgp", "estimand", "basis", "loss", "estimator"], dropna=False)["propensity_clip_rate"]
    .mean()
    .rename("propensity_clip_rate_mean")
    .reset_index()
)
summary_plugin = summary_plugin.merge(plugin_clip, on=["dgp", "estimand", "basis", "loss", "estimator"], how="left")

for estimand_name in ESTIMANDS:
    table_df = summary_plugin[summary_plugin["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "estimator"])
    display_table(table_df, caption=f"Plug-in logistic baseline: {estimand_name}")

# Direct comparison of the doubly robust arms: GRR ARW versus plug-in logistic AIPW.
combined_dr = pd.concat([main_compatible, main_plugin], ignore_index=True, sort=False)
combined_dr = combined_dr[combined_dr["estimator"].isin(["arw", "aipw"])].copy()
summary_dr = summarize_estimates(combined_dr, ["dgp", "estimand", "basis", "loss", "estimator"])
for estimand_name in ESTIMANDS:
    table_df = summary_dr[summary_dr["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "basis", "loss", "estimator"])
    display_table(table_df, caption=f"Doubly robust comparison (GRR ARW versus plug-in logistic AIPW): {estimand_name}")

## 2. Incompatible loss-link pairs

This subsection intentionally breaks dual linearity. The examples are `BKL loss + logit link`, `SQ loss + logit link`, and `UKL loss + linear link`. The model uses an RKHS basis and cross fitting. RA, RW/IPW, ARW, and TMLE are all displayed.

In [ ]:
INCOMPATIBLE_PAIRS = ["BKL loss + logit link", "SQ loss + logit link", "UKL loss + linear link"]
rows = []
for dgp_name in DGP_NAMES:
    for rep in range(N_REPLICATIONS):
        data = make_simulation_data(dgp_name, n=SAMPLE_SIZE, seed=200000 + 1009 * rep)
        for pair_name in INCOMPATIBLE_PAIRS:
            for estimand in ESTIMANDS:
                fit_rows = fit_one_incompatible(
                    data,
                    estimand=estimand,
                    pair_name=pair_name,
                    cross_fit=True,
                    lam=RIEZ_LAMBDA_MAIN,
                    basis_features=RIEZ_FEATURES,
                    folds=FOLDS,
                    random_state=rep,
                )
                for row in fit_rows:
                    row["dgp"] = dgp_name
                    row["replication"] = rep
                rows.extend(fit_rows)
incompatible_results = pd.DataFrame(rows)
incompatible_summary = summarize_estimates(incompatible_results, ["dgp", "estimand", "loss_link_pair", "estimator"])

for estimand_name in ESTIMANDS:
    table_df = incompatible_summary[incompatible_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "loss_link_pair", "estimator"])
    display_table(table_df, caption=f"Main simulation incompatible loss-link pairs: {estimand_name}")


In [ ]:
# Box plots for incompatible loss-link pairs. ATE and ATT are deliberately separated.
plot_df = incompatible_results[(incompatible_results["status"] == "ok") & (incompatible_results["estimator"].isin(["rw", "arw", "tmle"]))].copy()
plot_df["method"] = plot_df["loss_link_pair"] + " | " + plot_df["estimator"]
plot_df["squared_error_plot"] = plot_df["squared_error"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for dgp_name in DGP_NAMES:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["dgp"] == dgp_name)].copy()
        if panel_df.empty:
            print(f"No plot data for {estimand_name} ({dgp_name}).")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        ordered_methods = sorted(panel_df["method"].unique())
        box_data = [panel_df.loc[panel_df["method"] == m, "squared_error_plot"].dropna().to_numpy() for m in ordered_methods]
        box = ax.boxplot(box_data, tick_labels=[prettify_method(_m) for _m in ordered_methods], widths=PLOT_CONFIG["box_width"], patch_artist=True, showfliers=False)
        for patch in box["boxes"]:
            patch.set_facecolor("#9E9E9E")
            patch.set_alpha(0.75)
        for median in box["medians"]:
            median.set_color("black")
            median.set_linewidth(1.2)
        ax.set_yscale(PLOT_CONFIG["squared_error_y_scale"])
        ax.set_title(f"{estimand_name}: incompatible loss-link pairs ({dgp_name})", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Loss-link pair and estimator", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("Squared error", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="x", labelrotation=45, labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.tick_params(axis="y", labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
        fig.tight_layout()
        plt.show()


## 3. Regularization path under compatible loss-link pairs

The model is RKHS. We compare cross fitting with no cross fitting and vary the Riesz regularization parameter. The main plot uses MSE on the y-axis and λ on the x-axis.

In [ ]:
LAMBDA_GRID = np.array([1e-1, 3e-1, 1.0, 3.0], dtype=float)
rows = []
for dgp_name in DGP_NAMES:
    for rep in range(N_REPLICATIONS):
        data = make_simulation_data(dgp_name, n=SAMPLE_SIZE, seed=300000 + 1009 * rep)
        for cross_fit in [True, False]:
            for lam in LAMBDA_GRID:
                for loss_spec in COMPATIBLE_LOSSES:
                    for estimand in ESTIMANDS:
                        fit_rows = fit_one_grr(
                            data,
                            estimand=estimand,
                            loss_spec=loss_spec,
                            basis_kind="rkhs",
                            basis_mode="regressor",
                            cross_fit=cross_fit,
                            lam=float(lam),
                            basis_features=RIEZ_FEATURES,
                            degree=POLYNOMIAL_DEGREE,
                            folds=FOLDS,
                            estimators=ESTIMATORS_ALL,
                            random_state=rep,
                        )
                        for row in fit_rows:
                            row["dgp"] = dgp_name
                            row["replication"] = rep
                        rows.extend(fit_rows)
regularization_results = pd.DataFrame(rows)
regularization_summary = summarize_estimates(regularization_results, ["dgp", "estimand", "cross_fit", "lambda_riesz", "loss", "estimator"])

for estimand_name in ESTIMANDS:
    table_df = regularization_summary[regularization_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "cross_fit", "lambda_riesz", "loss", "estimator"])
    display_table(table_df, caption=f"Regularization path summary: {estimand_name}")


In [ ]:
# MSE paths. ATE and ATT, and each DGP, are plotted separately.
plot_df = regularization_summary[regularization_summary["estimator"] == "arw"].copy()
plot_df["mse_plot"] = plot_df["mse"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for dgp_name in DGP_NAMES:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["dgp"] == dgp_name)].copy()
        if panel_df.empty:
            print(f"No path data for {estimand_name} ({dgp_name}).")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        for (loss, cross_fit), g in panel_df.groupby(["loss", "cross_fit"]):
            g = g.sort_values("lambda_riesz")
            linestyle = "-" if bool(cross_fit) else ":"
            label = f"{label_of(loss)} (cross_fit={bool(cross_fit)})"
            ax.plot(g["lambda_riesz"], g["mse_plot"], marker="o", markersize=PLOT_CONFIG["marker_size"], linewidth=PLOT_CONFIG["line_width"], linestyle=linestyle, color=METHOD_COLORS.get(loss), label=label)
        ax.set_xscale("log")
        ax.set_title(f"Regularization path in {estimand_name} estimation. {dgp_name}", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Riesz regularization parameter λ", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("MSE", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="both", labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(alpha=PLOT_CONFIG["grid_alpha"])
        ax.legend(fontsize=PLOT_CONFIG["legend_fontsize"], ncol=2)
        fig.tight_layout()
        plt.show()
